In [29]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 2.2.3
NumPy version: 2.1.3


In [30]:
# Load the raw orders dataset
df = pd.read_csv("/content/orders.csv")

# Display basic information
print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Show first 5 records
df.head()

Dataset loaded successfully!
Rows: 32313
Columns: 8


,order_id,created_at,website_session_id,user_id,primary_product_id,items_purchased,price_usd,cogs_usd
0,1,2012-03-19 10:42:46,20,20,1,1,49.99,19.49
1,2,2012-03-19 19:27:37,104,104,1,1,49.99,19.49
2,3,2012-03-20 06:44:45,147,147,1,1,49.99,19.49
3,4,2012-03-20 09:41:45,160,160,1,1,49.99,19.49
4,5,2012-03-20 11:28:15,177,177,1,1,49.99,19.49


In [31]:
# BEFORE CLEANING - DATA PROFILE

print("===== DATASET INFORMATION =====")
print(df.info())

print("\n===== MISSING VALUES =====")
print(df.isnull().sum())

print("\n===== DUPLICATE RECORDS =====")
print("Duplicate rows:", df.duplicated().sum())

print("\n===== DATA TYPES =====")
print(df.dtypes)

print("\n===== BASIC STATISTICS =====")
display(df.describe(include="all"))

===== DATASET INFORMATION =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32313 entries, 0 to 32312
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            32313 non-null  int64  
 1   created_at          32313 non-null  object 
 2   website_session_id  32313 non-null  int64  
 3   user_id             32313 non-null  int64  
 4   primary_product_id  32313 non-null  int64  
 5   items_purchased     32313 non-null  int64  
 6   price_usd           32313 non-null  float64
 7   cogs_usd            32313 non-null  float64
dtypes: float64(2), int64(5), object(1)
memory usage: 2.0+ MB
None

===== MISSING VALUES =====
order_id              0
created_at            0
website_session_id    0
user_id               0
primary_product_id    0
items_purchased       0
price_usd             0
cogs_usd              0
dtype: int64

===== DUPLICATE RECORDS =====
Duplicate rows: 0

===== DATA TYPES ===

,order_id,created_at,website_session_id,user_id,primary_product_id,items_purchased,price_usd,cogs_usd
count,32313.000000,32313,32313.000000,32313.000000,32313.000000,32313.000000,32313.000000,32313.000000
unique,NaN,32299,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,2014-07-08 15:02:38,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN
mean,16157.000000,NaN,258292.288707,215691.622629,1.392474,1.238666,59.991636,22.355406
std,9328.103961,NaN,132427.649842,108402.203189,0.732277,0.426274,17.808771,6.238621
min,1.000000,NaN,20.000000,13.000000,1.000000,1.000000,29.990000,9.490000
25%,8079.000000,NaN,144828.000000,124135.000000,1.000000,1.000000,49.990000,19.490000
50%,16157.000000,NaN,263554.000000,221461.000000,1.000000,1.000000,49.990000,19.490000
75%,24235.000000,NaN,374799.000000,310542.000000,2.000000,1.000000,59.990000,22.490000


In [32]:
# DATE VALIDITY CHECK

print("===== DATE VALIDITY CHECK =====")

# Convert created_at to datetime
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")

print("Invalid dates:", df["created_at"].isna().sum())
print("Earliest date:", df["created_at"].min())
print("Latest date:", df["created_at"].max())


# OUTLIER CHECK USING IQR

print("\n===== OUTLIER CHECK =====")

numeric_columns = [
    "items_purchased",
    "price_usd",
    "cogs_usd"
]

for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ]

    print(f"\n{col}")
    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)
    print("Outliers:", len(outliers))

===== DATE VALIDITY CHECK =====
Invalid dates: 0
Earliest date: 2012-03-19 10:42:46
Latest date: 2015-03-19 05:38:31

===== OUTLIER CHECK =====

items_purchased
Lower bound: 1.0
Upper bound: 1.0
Outliers: 7712

price_usd
Lower bound: 34.99
Upper bound: 74.99000000000001
Outliers: 8246

cogs_usd
Lower bound: 14.989999999999998
Upper bound: 26.99
Outliers: 9623


In [33]:
# UNDERSTANDING POTENTIAL OUTLIERS

print("===== ITEMS PURCHASED =====")
print(df["items_purchased"].value_counts().sort_index())

print("\n===== PRICE VALUES =====")
print(df["price_usd"].value_counts().sort_index())

print("\n===== COGS VALUES =====")
print(df["cogs_usd"].value_counts().sort_index())

print("\n===== INVALID / IMPOSSIBLE VALUES =====")
print("Items purchased <= 0:", (df["items_purchased"] <= 0).sum())
print("Price <= 0:", (df["price_usd"] <= 0).sum())
print("COGS <= 0:", (df["cogs_usd"] <= 0).sum())


===== ITEMS PURCHASED =====
items_purchased
1    24601
2     7712
Name: count, dtype: int64

===== PRICE VALUES =====
price_usd
29.99       534
45.99      2039
49.99     18104
59.99      3924
75.98       662
79.98      3142
89.98       680
95.98      2036
105.98      248
109.98      944
Name: count, dtype: int64

===== COGS VALUES =====
cogs_usd
9.49       534
14.49     2039
19.49    18104
22.49     3924
23.98      662
28.98     3142
31.98      680
33.98     2036
36.98      248
41.98      944
Name: count, dtype: int64

===== INVALID / IMPOSSIBLE VALUES =====
Items purchased <= 0: 0
Price <= 0: 0
COGS <= 0: 0


In [34]:
# ==========================================
# DATA CLEANING & FEATURE ENGINEERING
# ==========================================

# Create a copy of the raw dataset
clean_df = df.copy()

# 1. Remove exact duplicate records
clean_df = clean_df.drop_duplicates()

# 2. Convert date column to proper datetime
clean_df["created_at"] = pd.to_datetime(
    clean_df["created_at"],
    errors="coerce"
)

# 3. Convert numeric columns to numeric datatype
numeric_columns = [
    "order_id",
    "website_session_id",
    "user_id",
    "primary_product_id",
    "items_purchased",
    "price_usd",
    "cogs_usd"
]

for col in numeric_columns:
    clean_df[col] = pd.to_numeric(
        clean_df[col],
        errors="coerce"
    )

# 4. Remove records with invalid essential values
clean_df = clean_df.dropna(
    subset=[
        "order_id",
        "created_at",
        "items_purchased",
        "price_usd",
        "cogs_usd"
    ]
)

# 5. Keep only valid positive business values
clean_df = clean_df[
    (clean_df["items_purchased"] > 0) &
    (clean_df["price_usd"] > 0) &
    (clean_df["cogs_usd"] > 0)
].copy()

# ==========================================
# FEATURE ENGINEERING
# ==========================================

# 6. Extract year and month from date
clean_df["year"] = clean_df["created_at"].dt.year
clean_df["month"] = clean_df["created_at"].dt.month

# 7. Create month name
clean_df["month_name"] = clean_df["created_at"].dt.month_name()

# 8. Calculate revenue
clean_df["revenue_usd"] = (
    clean_df["items_purchased"] *
    clean_df["price_usd"]
)

# 9. Calculate total cost
clean_df["total_cost_usd"] = (
    clean_df["items_purchased"] *
    clean_df["cogs_usd"]
)

# 10. Calculate profit
clean_df["profit_usd"] = (
    clean_df["revenue_usd"] -
    clean_df["total_cost_usd"]
)

# 11. Calculate profit margin percentage
clean_df["profit_margin_pct"] = (
    clean_df["profit_usd"] /
    clean_df["revenue_usd"]
) * 100

# ==========================================
# FINAL CHECK
# ==========================================

print("===== CLEANING RESULTS =====")
print("Original rows:", len(df))
print("Clean rows:", len(clean_df))
print("Rows removed:", len(df) - len(clean_df))

print("\nMissing values after cleaning:")
print(clean_df.isnull().sum())

print("\nDuplicate rows after cleaning:",
      clean_df.duplicated().sum())

print("\nFinal data types:")
print(clean_df.dtypes)

print("\nFinal columns:")
print(clean_df.columns.tolist())

print("\nSample cleaned data:")
display(clean_df.head())

===== CLEANING RESULTS =====
Original rows: 32313
Clean rows: 32313
Rows removed: 0

Missing values after cleaning:
order_id              0
created_at            0
website_session_id    0
user_id               0
primary_product_id    0
items_purchased       0
price_usd             0
cogs_usd              0
year                  0
month                 0
month_name            0
revenue_usd           0
total_cost_usd        0
profit_usd            0
profit_margin_pct     0
dtype: int64

Duplicate rows after cleaning: 0

Final data types:
order_id                       int64
created_at            datetime64[ns]
website_session_id             int64
user_id                        int64
primary_product_id             int64
items_purchased                int64
price_usd                    float64
cogs_usd                     float64
year                           int32
month                          int32
month_name                    object
revenue_usd                  float64
total_cost_usd

,order_id,created_at,website_session_id,user_id,primary_product_id,items_purchased,price_usd,cogs_usd,year,month,month_name,revenue_usd,total_cost_usd,profit_usd,profit_margin_pct
0,1,2012-03-19 10:42:46,20,20,1,1,49.99,19.49,2012,3,March,49.99,19.49,30.5,61.012202
1,2,2012-03-19 19:27:37,104,104,1,1,49.99,19.49,2012,3,March,49.99,19.49,30.5,61.012202
2,3,2012-03-20 06:44:45,147,147,1,1,49.99,19.49,2012,3,March,49.99,19.49,30.5,61.012202
3,4,2012-03-20 09:41:45,160,160,1,1,49.99,19.49,2012,3,March,49.99,19.49,30.5,61.012202
4,5,2012-03-20 11:28:15,177,177,1,1,49.99,19.49,2012,3,March,49.99,19.49,30.5,61.012202


In [35]:
# ==========================================
# BEFORE vs AFTER CLEANING SUMMARY
# ==========================================

print("===== BEFORE vs AFTER CLEANING =====")

summary = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Missing Values",
        "Duplicate Rows"
    ],
    "Before Cleaning": [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum()
    ],
    "After Cleaning": [
        clean_df.shape[0],
        clean_df.shape[1],
        clean_df.isnull().sum().sum(),
        clean_df.duplicated().sum()
    ]
})

display(summary)

===== BEFORE vs AFTER CLEANING =====


,Metric,Before Cleaning,After Cleaning
0,Total Rows,32313,32313
1,Total Columns,8,15
2,Missing Values,0,0
3,Duplicate Rows,0,0


In [36]:
# ==========================================
# EXPORT CLEAN DATASET
# ==========================================

output_file = "/content/clean_dataset.csv"

clean_df.to_csv(
    output_file,
    index=False
)

print("Clean dataset exported successfully!")
print("File:", output_file)
print("Rows:", len(clean_df))
print("Columns:", len(clean_df.columns))

Clean dataset exported successfully!
File: /content/clean_dataset.csv
Rows: 32313
Columns: 15
